# PyTorch 网络层详解 - 总论

本 Notebook 是 PyTorch 网络层系列教程的第一部分，建立对 PyTorch 神经网络组件体系的整体认知。

## 📚 系列结构

- **第1部分：总论**（本文）—— 建立全局框架，理解核心概念与设计哲学
- **第2部分：有参数层**—— 卷积、线性、归一化、嵌入、序列层、Transformer 核心组件
- **第3部分：无参数层**—— 激活函数、池化、Dropout 等无参数但有状态的操作
- **第4部分：容器**—— Sequential、ModuleList、ModuleDict、ParameterList/ParameterDict
- **第5部分：综合应用**—— 完整训练流程 + 高级技巧（自定义层、权重初始化、梯度裁剪等）

## 🗺️ 本文脉络

本文按 **"由表及里、由粗到细"** 的逻辑组织，帮助读者循序渐进地建立知识体系：

**第一部分：宏观定位 —— torch.nn 包的整体结构**（第0-1节）

理解层、容器、模型、损失函数四大概念的层次关系。

↓

**第二部分：核心机制 —— nn.Module 的底层设计原理**（第2-3节）

理解为什么所有组件都继承 nn.Module，以及它与 functional 的微妙区别。

↓

**第三部分：状态分类 —— 网络层的三种形态**（第4-5节）

有状态层、无参数层、无状态函数，以及单一层 vs 复合层。

↓

**第四部分：存储原理 —— 状态是如何被管理的**（第6节）

从"什么是状态"出发，介绍存储机制（__setattr__ 分流 → 三个私有容器），以及 state_dict 的用法。

↓

**第五部分：能力全览 —— nn.Module 的所有可用方法**（第7节）

十大类方法：数据访问、模块管理、注册API、模式控制、设备迁移、梯度操作、序列化、工具方法、钩子、魔法方法。

↓

**第六部分：总结升华 —— 速查表 + 选择指南**（第8节）

梳理核心概念，给出不同场景下的推荐方案。

**整体逻辑**：先看全局（宏观定位）→ 再挖底层（核心机制）→ 再分门别类（状态分类）→ 再深入存储（存储原理）→ 再遍历能力（方法全览）→ 最后提炼总结（速查指南）。层层递进，从认知到应用，从原理到实践。

In [1]:
# ============================================================
# 1. 导入必要的库
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import warnings
import copy
warnings.filterwarnings('ignore')

# 设置随机种子，保证实验结果可复现
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.7.1+cu118
CUDA available: True


## 0. 四个核心概念

在 PyTorch 中，`nn.Module` 的子类根据**用途**可以分为四个层次：

| 概念 | 含义 | 生活类比 | 代码示例 |
|------|------|----------|----------|
| **层（Layer）** | 执行特定数据变换的**最小计算单元** | 工厂里的一台机器（如切割机） | `nn.Linear`、`nn.Conv2d`、`nn.ReLU` |
| **容器（Container）** | 用来**组织和管理多个层**的结构 | 工厂里的一条流水线（把多台机器串起来） | `nn.Sequential`、`nn.ModuleList` |
| **模型（Model）** | 用户定义的**完整神经网络**，由层和容器组合而成 | 整个工厂（包含多条流水线和机器） | 自定义的 `class MyModel(nn.Module)` |
| **损失函数（Loss）** | 计算**预测值与真实值之间的差异** | 工厂里的质检员（检查产品质量） | `nn.MSELoss`、`nn.CrossEntropyLoss` |

**关键理解**：

| 概念 | 是否参与前向传播 | 是否有可训练参数 | 是否放入模型内部 |
|------|-----------------|-----------------|-----------------|
| **层** | ✅ 是（数据流过它） | 有的有，有的无 | ✅ 是 |
| **容器** | ✅ 是（数据流过它） | ❌ 否（本身无参） | ✅ 是 |
| **模型** | ✅ 是（数据流过它） | 取决于内部设计 | ❌ 否（它是最外层） |
| **损失函数** | ❌ 否（不处理数据变换，只计算差异） | ❌ 否 | ❌ 否（在训练循环中调用） |

> 💡 **一句话**：在 PyTorch 里，**层、容器、模型、损失函数都是 `nn.Module` 的子类**，只是使用场景和组织层次不同。**数据（`torch.Tensor`）不是 Module**，它是在各模块之间流动的待加工"材料"。

In [2]:
# ============================================================
# 示例：验证四大类都是 nn.Module 的子类
# ============================================================

print("=" * 60)
print("【验证：层、容器、模型、损失函数都是 nn.Module 的子类】")
print("=" * 60)

# 1. 层 - nn.Linear（全连接层）
# 这是最基础的神经网络层之一，执行 y = x @ W.T + b 的线性变换
linear = nn.Linear(10, 5)
print(f"\n1. nn.Linear 是 nn.Module 的子类: {isinstance(linear, nn.Module)}")

# 2. 容器 - nn.Sequential（顺序容器）
# Sequential 按顺序执行其中的所有子模块，本身不包含可训练参数
seq = nn.Sequential(nn.Linear(10, 5))
print(f"2. nn.Sequential 是 nn.Module 的子类: {isinstance(seq, nn.Module)}")

# 3. 模型 - 自定义 MyModel
# 用户自定义的完整神经网络，通过继承 nn.Module 获得所有核心能力
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 5)  # 在模型中嵌套层
    def forward(self, x):
        return self.fc(x)
model = MyModel()
print(f"3. 自定义模型是 nn.Module 的子类: {isinstance(model, nn.Module)}")

# 4. 损失函数 - nn.MSELoss（均方误差损失）
# 损失函数也是 nn.Module 的子类，用于计算预测值与真实值的差异
loss = nn.MSELoss()
print(f"4. nn.MSELoss 是 nn.Module 的子类: {isinstance(loss, nn.Module)}")

# 额外验证：数据 Tensor 不是 nn.Module
# Tensor 是数据载体，不是可执行的模块，因此不是 nn.Module 的子类
x = torch.randn(3, 10)
print(f"\n5. torch.Tensor 是 nn.Module 的子类: {isinstance(x, nn.Module)}")

【验证：层、容器、模型、损失函数都是 nn.Module 的子类】

1. nn.Linear 是 nn.Module 的子类: True
2. nn.Sequential 是 nn.Module 的子类: True
3. 自定义模型是 nn.Module 的子类: True
4. nn.MSELoss 是 nn.Module 的子类: True

5. torch.Tensor 是 nn.Module 的子类: False


## 1. `torch.nn` 包结构

```
torch.nn
├── 网络层（Layers）              ← 本系列教程的核心
│   ├── nn.Linear                # 全连接层
│   ├── nn.Conv2d                # 卷积层
│   ├── nn.BatchNorm2d           # 批归一化
│   ├── nn.ReLU                  # 激活层
│   ├── nn.Dropout               # Dropout层
│   ├── nn.Embedding             # 嵌入层
│   └── nn.LSTM / nn.GRU         # 循环层（复合层）
│
├── 容器（Containers）            ← 第4部分详解
│   ├── nn.Sequential
│   ├── nn.ModuleList
│   └── nn.ModuleDict
│
├── 损失函数（Loss Functions）
│   ├── nn.MSELoss
│   ├── nn.CrossEntropyLoss
│   └── nn.BCELoss
│
└── 功能函数（Functional）        ← 无状态的函数版本
    └── nn.functional (F)
```

## 2. `nn.Module` 核心能力

所有继承 `nn.Module` 的组件自动获得以下能力：

| 能力 | 说明 |
|------|------|
| **参数管理** | 自动追踪所有可训练参数 |
| **设备迁移** | `.to(device)` 一键移到 GPU |
| **保存加载** | `.state_dict()` / `.load_state_dict()` |
| **模式切换** | `.train()` / `.eval()` |
| **模块嵌套** | Module 里可以放 Module |

In [3]:
# ============================================================
# 示例：nn.Module 核心能力
# ============================================================

print("=" * 60)
print("【nn.Module 核心能力演示】")
print("=" * 60)

model = nn.Linear(10, 5)

# 1. 参数管理：自动追踪所有可训练参数
# parameters() 方法递归收集所有 nn.Parameter 类型的张量
total_params = sum(p.numel() for p in model.parameters())
print(f"\n1. 参数管理: {total_params} 个参数")
print(f"   weight shape: {model.weight.shape}, bias shape: {model.bias.shape}")

# 2. 设备迁移：一键将模型移到指定设备
# 注意：此操作是 in-place 的，会修改模型内部的张量存储位置
print(f"\n2. 设备迁移: {next(model.parameters()).device}")
print(f"   调用 .cuda() 前需确保有 CUDA 设备")

# 3. state_dict：导出模型的所有状态（参数 + buffer）
# state_dict 是一个 OrderedDict，键是参数名，值是张量
print(f"\n3. state_dict 键: {list(model.state_dict().keys())}")

# 4. 模式切换：训练模式 vs 推理模式
# training 标志会影响 Dropout、BatchNorm 等层的行为
print(f"\n4. 模式切换: training={model.training}")
model.eval()  # 切换到推理模式，禁用 Dropout，使用 BN 的全局统计量
print(f"   调用 eval() 后: training={model.training}")
model.train()  # 切换回训练模式
print(f"   调用 train() 后: training={model.training}")

# 5. 模块嵌套验证：Module 可以包含其他 Module
# 这种嵌套结构允许构建任意深度的神经网络
class NestedModule(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 10)  # 子模块1
        self.fc2 = nn.Linear(10, 5)   # 子模块2
nested = NestedModule()
print(f"\n5. 模块嵌套: 总参数 = {sum(p.numel() for p in nested.parameters())}")
print(f"   子模块列表: {list(nested._modules.keys())}")

【nn.Module 核心能力演示】

1. 参数管理: 55 个参数
   weight shape: torch.Size([5, 10]), bias shape: torch.Size([5])

2. 设备迁移: cpu
   调用 .cuda() 前需确保有 CUDA 设备

3. state_dict 键: ['weight', 'bias']

4. 模式切换: training=True
   调用 eval() 后: training=False
   调用 train() 后: training=True

5. 模块嵌套: 总参数 = 165
   子模块列表: ['fc1', 'fc2']


## 3. `nn.Module` vs `nn.functional`

我们已经了解了 `nn.Module` 是什么以及它的核心能力。但在实际使用 PyTorch 构建网络时，你会发现实现同一个操作常常有两套 API：一套是 `nn.Module` 子类（如 `nn.ReLU`），另一套是 `nn.functional` 函数（如 `F.relu`）。

这两者有什么区别？什么时候用哪个？这是新手最容易混淆的点。

**核心区别**：
- `nn.Module` 子类是**有状态的对象**，内部维护可训练参数和运行时状态
- `nn.functional` 函数是**无状态的纯函数**，每次调用都独立计算，不维护任何内部状态

| 对比维度 | `nn.Module` 子类 | `nn.functional` 函数 |
|----------|------------------|----------------------|
| **本质** | 类（有状态对象） | 纯函数（无状态） |
| **参数** | 内部自己维护 | 需要外部传入 |
| **能否放入 Sequential** | ✅ 能 | ❌ 不能 |
| **典型例子** | `nn.Linear`, `nn.Conv2d`, `nn.ReLU` | `F.linear`, `F.conv2d`, `F.relu` |

**什么时候用哪个？**

| 场景 | 推荐 | 原因 |
|------|------|------|
| 有可训练参数（Linear、Conv） | `nn.Module` 子类 | 参数需要被优化器追踪 |
| 需要放入 `Sequential` | `nn.Module` 子类 | Sequential 只接受 Module |
| 自定义 forward 里的临时计算 | `F` 函数 | 更轻量，无状态开销 |
| 需要 train/eval 模式切换（Dropout、BN） | `nn.Module` 子类 | 自动处理模式切换 |

In [4]:
# ============================================================
# 示例：nn.Module vs nn.functional —— 同一功能，两种写法
# ============================================================

print("=" * 60)
print("【nn.Module vs nn.functional —— 同一功能，两种写法】")
print("=" * 60)

# 统一输入张量，全程复用，减少重复创建
x = torch.randn(3, 10)
print(f"全局统一输入 x.shape = {x.shape}\n")

print("-" * 40)
print("对比1：全连接层（带可训练参数）")
print("-" * 40)

# 1. nn.Module 方式：实例化时内部自动随机初始化权重、偏置
# 这是最常用的方式，因为参数会被自动管理并注册到优化器中
linear_module = nn.Linear(10, 5)
out_module = linear_module(x)
print(f"【nn.Linear（有状态模块）】")
print(f"  实例化时自动随机初始化内置可训练参数：weight={linear_module.weight.shape}, bias={linear_module.bias.shape}")
print(f"  调用语法：out = linear_module(x)")
print(f"  计算输出形状：{out_module.shape}\n")

# 2. F.linear 函数方式：纯计算，参数必须手动外部随机定义
# 这种方式在自定义层中偶尔使用，但一般不推荐直接用于构建网络
w = torch.randn(5, 10)
b = torch.randn(5)
out_func = F.linear(x, w, b)
print(f"【F.linear（无状态纯函数）】")
print(f"  参数需要手动随机创建：w={w.shape}, b={b.shape}")
print(f"  调用语法：out = F.linear(x, weight, bias)")
print(f"  计算输出形状：{out_func.shape}")

print("\n" + "-" * 40)
print("对比2：ReLU 激活（无参数，仅状态区分）")
print("-" * 40)

# 1. nn.ReLU 模块：拥有training状态，可放入Sequential
# 虽然 ReLU 本身没有可训练参数，但作为 Module 子类可以放入容器
relu_module = nn.ReLU()
out_relu_mod = relu_module(x)
print(f"【nn.ReLU（模块）】")
print(f"  自带 train/eval 状态标记，支持容器嵌套，无权重参数")
print(f"  调用语法：out = relu_module(x)")
print(f"  计算输出形状：{out_relu_mod.shape}\n")

# 2. F.relu 函数：无状态，仅即时计算
# 函数版本更轻量，适合在自定义 forward 中临时调用
out_relu_func = F.relu(x)
print(f"【F.relu（纯函数）】")
print(f"  无状态属性，无法放入Sequential，无任何参数")
print(f"  调用语法：out = F.relu(x)")
print(f"  计算输出形状：{out_relu_func.shape}")

# 额外验证：能否放入 Sequential
# Sequential 要求传入的是 nn.Module 子类实例，不能是函数
print("\n" + "-" * 40)
print("对比3：能否放入 Sequential")
print("-" * 40)
try:
    # nn.Module 子类可以正常放入 Sequential
    seq = nn.Sequential(nn.Linear(10, 5), nn.ReLU())
    print("【验证】nn.ReLU 可以放入 Sequential: ✅ 成功")
except Exception as e:
    print(f"【验证】nn.ReLU 放入 Sequential 失败: {e}")

try:
    # F.relu 是函数，放入 Sequential 虽然可行但不推荐
    seq = nn.Sequential(nn.Linear(10, 5), F.relu)
    print("【验证】F.relu 可以放入 Sequential: ⚠️ 可以但需要传入函数对象")
except Exception as e:
    print(f"【验证】F.relu 放入 Sequential 失败: {e}")

print("\n" + "=" * 60)
print("✅ 核心结论:")
print("  1. nn.Module 子类：实例化自动随机初始化权重，内置存储参数/运行状态，可作为网络层放入Sequential，标准搭建模型使用")
print("  2. nn.functional 函数：不会自动创建参数，权重需要手动随机生成，无存储、无状态，仅做临时即时计算，适合forward内部简单运算")
print("  3. F.relu 虽可传入 Sequential，但推荐使用 nn.ReLU 以保持一致性")
print("=" * 60)

【nn.Module vs nn.functional —— 同一功能，两种写法】
全局统一输入 x.shape = torch.Size([3, 10])

----------------------------------------
对比1：全连接层（带可训练参数）
----------------------------------------
【nn.Linear（有状态模块）】
  实例化时自动随机初始化内置可训练参数：weight=torch.Size([5, 10]), bias=torch.Size([5])
  调用语法：out = linear_module(x)
  计算输出形状：torch.Size([3, 5])

【F.linear（无状态纯函数）】
  参数需要手动随机创建：w=torch.Size([5, 10]), b=torch.Size([5])
  调用语法：out = F.linear(x, weight, bias)
  计算输出形状：torch.Size([3, 5])

----------------------------------------
对比2：ReLU 激活（无参数，仅状态区分）
----------------------------------------
【nn.ReLU（模块）】
  自带 train/eval 状态标记，支持容器嵌套，无权重参数
  调用语法：out = relu_module(x)
  计算输出形状：torch.Size([3, 10])

【F.relu（纯函数）】
  无状态属性，无法放入Sequential，无任何参数
  调用语法：out = F.relu(x)
  计算输出形状：torch.Size([3, 10])

----------------------------------------
对比3：能否放入 Sequential
----------------------------------------
【验证】nn.ReLU 可以放入 Sequential: ✅ 成功
【验证】F.relu 放入 Sequential 失败: torch.nn.functional.relu is not a Module subclass

✅ 核心结论:
 

## 4. 网络层的两种类型：单一层 vs 复合层

### 4.1 单一层（原子操作层）

执行**单一、不可再分**的数学操作：

| 类别 | 层 | 有无参数 |
|------|-----|---------|
| **卷积** | `nn.Conv2d` | ✅ 有 |
| **线性** | `nn.Linear` | ✅ 有 |
| **归一化** | `nn.BatchNorm2d` | ✅ 有 |
| **汇聚** | `nn.MaxPool2d`、`nn.AvgPool2d` | ❌ 无 |
| **激活** | `nn.ReLU`、`nn.Sigmoid` | ❌ 无 |
| **Dropout** | `nn.Dropout` | ❌ 无 |
| **嵌入** | `nn.Embedding` | ✅ 有 |

### 4.2 复合层（组合/模块化层）

由**多个子层组合而成**：

| 类别 | 层 | 内部组成 |
|------|-----|---------|
| **RNN** | `nn.RNN` | 循环 + 激活（tanh/relu） |
| **LSTM** | `nn.LSTM` | 4个门控（遗忘门、输入门、细胞门、输出门） |
| **GRU** | `nn.GRU` | 2个门控（更新门、重置门） |
| **Transformer** | `nn.MultiheadAttention` | Q/K/V 投影 + 注意力计算 |
| **Transformer** | `nn.TransformerEncoder` | 多头注意力 + FFN + LayerNorm + 残差连接 |

In [5]:
# ============================================================
# 验证 LSTM 是复合层（内部包含多个 Linear）
# ============================================================

print("=" * 60)
print("【验证：LSTM 内部包含多个 Linear】")
print("=" * 60)

# 创建一个单层 LSTM
# input_size=10: 输入特征维度, hidden_size=20: 隐藏状态维度
lstm = nn.LSTM(input_size=10, hidden_size=20, num_layers=1)
# 创建对照用的全连接层
linear = nn.Linear(10, 20)

print("\n1. LSTM 参数形状:")
# LSTM 内部有 4 组门控参数，每组对应一个 Linear 层
for name, param in lstm.named_parameters():
    print(f"   {name}: {param.shape}")

print("\n2. 验证单门控权重形状:")
# weight_ih_l0 是输入到隐藏的权重，shape = (4 * hidden_size, input_size)
# 其中 4 对应 4 个门控：输入门、遗忘门、细胞门、输出门
print(f"   weight_ih_l0 单门控权重形状: ({lstm.hidden_size}, {lstm.input_size})")
print(f"   Linear(10,20) 权重形状: {linear.weight.shape}")
print(f"   → 一致: ({lstm.hidden_size}, {lstm.input_size})")

【验证：LSTM 内部包含多个 Linear】

1. LSTM 参数形状:
   weight_ih_l0: torch.Size([80, 10])
   weight_hh_l0: torch.Size([80, 20])
   bias_ih_l0: torch.Size([80])
   bias_hh_l0: torch.Size([80])

2. 验证单门控权重形状:
   weight_ih_l0 单门控权重形状: (20, 10)
   Linear(10,20) 权重形状: torch.Size([20, 10])
   → 一致: (20, 10)


## 5. 有状态 vs 无状态（三种形态）

在 PyTorch 中，"状态"指的是模块是否拥有**可训练参数**或**运行时状态**（如 train/eval 模式）。根据这个标准，网络层可以分为三种形态：

**有状态层**：`nn.Linear`, `nn.Conv2d`——有可训练参数，能放入 Sequential ✅

**无参数层**：`nn.ReLU`, `nn.Dropout`——无参数，但有 train/eval 模式，能放入 Sequential ✅

**无状态函数**：`F.relu`, `F.conv2d`——纯函数，无参数无状态，不能放入 Sequential ❌

> 📌 **关键理解**："有状态"不一定是"有参数"。`nn.Dropout` 虽然没参数，但在 train/eval 模式下行为不同，所以它是有状态的 `nn.Module`。

---

🔍 **思考**：上面我们区分了三种形态——有状态层（如 Linear）、无参数层（如 Dropout）、无状态函数（如 F.relu）。

现在一个关键问题浮出水面：**这些"状态"到底存在哪里？**

- `nn.Linear` 的 weight 和 bias 存在哪？
- `nn.Dropout` 的 training 状态存在哪？
- `nn.BatchNorm` 的 running_mean 存在哪？

答案是：`nn.Module` 通过一套精密的存储机制，将不同类型的"状态"自动分类管理。下面我们就来揭开这个机制的面纱。

In [6]:
# ============================================================
# 示例：三种形态对比
# ============================================================

print("=" * 60)
print("【三种形态：有状态层、无参数层、无状态函数】")
print("=" * 60)

# 1. 有状态层（有参数）
# nn.Linear 包含可训练的 weight 和 bias，是典型的有参数层
linear = nn.Linear(10, 5)
print(f"\n1. nn.Linear (有状态层):")
print(f"   有参数: {sum(p.numel() for p in linear.parameters()) > 0}")
print(f"   参数量: {sum(p.numel() for p in linear.parameters())}")
print(f"   可放入 Sequential: {isinstance(linear, nn.Module)}")

# 2. 无参数层（有状态，无参数）
# nn.ReLU 没有可训练参数，但作为 Module 子类拥有 training 状态
relu = nn.ReLU()
print(f"\n2. nn.ReLU (无参数层):")
print(f"   有参数: {len(list(relu.parameters())) == 0}")
print(f"   可放入 Sequential: {isinstance(relu, nn.Module)}")
print(f"   有 train/eval 状态: {hasattr(relu, 'training')}")

# 3. 无状态函数
# F.relu 是纯函数，没有任何内部状态
print(f"\n3. F.relu (无状态函数):")
print(f"   是函数对象: {callable(F.relu)}")
print(f"   有 training 状态: {hasattr(F.relu, 'training')}")

# 4. Dropout 演示状态差异
# Dropout 在训练模式下随机置零，推理模式下保持不变
dropout = nn.Dropout(p=0.5)
x = torch.ones(1, 10)

dropout.train()  # 训练模式：随机丢弃
out_train = dropout(x)
dropout.eval()   # 推理模式：保持不变（缩放）
out_eval = dropout(x)

print(f"\n4. nn.Dropout 状态差异验证:")
print(f"   有参数: {len(list(dropout.parameters())) == 0}")
print(f"   训练模式输出和推理模式输出不同: {bool((out_train != out_eval).any())}")
print(f"   训练模式输出均值: {out_train.mean().item():.4f}")
print(f"   推理模式输出均值: {out_eval.mean().item():.4f}")
print(f"   → 说明 nn.Dropout 虽然没有参数，但有状态")

【三种形态：有状态层、无参数层、无状态函数】

1. nn.Linear (有状态层):
   有参数: True
   参数量: 55
   可放入 Sequential: True

2. nn.ReLU (无参数层):
   有参数: True
   可放入 Sequential: True
   有 train/eval 状态: True

3. F.relu (无状态函数):
   是函数对象: True
   有 training 状态: False

4. nn.Dropout 状态差异验证:
   有参数: True
   训练模式输出和推理模式输出不同: True
   训练模式输出均值: 0.8000
   推理模式输出均值: 1.0000
   → 说明 nn.Dropout 虽然没有参数，但有状态


## 6. 存储原理：状态是如何被管理的

上一节我们提出了一个问题：状态到底存在哪里？本节就来回答这个问题。

### 6.1 什么是"状态"？

在 PyTorch 中，**状态就是模型在训练或推理时所需的所有持久化数据**，包括可训练参数（如权重和偏置）和非训练统计量（如 BatchNorm 的均值和方差），它们通过 `state_dict()` 统一保存，用于完整恢复模型的参数配置，从而实现模型的保存、加载和迁移学习。

具体包含三类内容：

| 类型 | 说明 | 示例 |
|------|------|------|
| **可训练参数** | 在训练过程中被优化器更新的张量 | `nn.Linear` 的 `weight` 和 `bias` |
| **运行时状态** | 影响前向传播行为的模式标志 | `training` 标志（影响 Dropout/BN） |
| **统计缓存** | 不参与梯度计算但需要持久保存的张量 | `BatchNorm` 的 `running_mean` 和 `running_var` |

这些状态**存储**在三个私有容器中（`_parameters`、`_modules`、`_buffers`）。而 `state_dict()` 是一个**收集器方法**，在需要的时候（保存模型、检查权重等）从这些容器中遍历并导出所有状态，返回一个 `OrderedDict`。


**使用方式**：

```python
# 导出状态（用于保存或查看）
sd = model.state_dict()

# 保存到文件
torch.save(sd, 'model.pth')

# 加载状态
model.load_state_dict(torch.load('model.pth'))
```

---

### 6.2 三个私有存储容器

| 私有容器 | 存储内容 | state_dict 是否包含 |
|----------|----------|---------------------|
| `_parameters` | `nn.Parameter` 可训练权重/偏置 | ✅ 全部保存 |
| `_modules` | 子 `nn.Module` | 递归收集子模块参数 |
| `_buffers` | 注册缓存张量（如 running_mean） | ✅ persistent=True 则保存 |

---

### 6.3 Parameter vs Buffer

在理解了三个容器之后，我们来深入对比 Parameter 和 Buffer 这两个核心概念：

| 维度 | Parameter | Buffer |
|------|-----------|--------|
| **是否可训练** | ✅ 是，有梯度 | ❌ 否，无梯度 |
| **优化器更新** | ✅ 会被更新 | ❌ 不会被更新 |
| **是否保存** | ✅ 保存到 state_dict | ✅ 保存到 state_dict |
| **典型用途** | 权重、偏置 | 均值、方差 |
| **存储位置** | `_parameters` 字典 | `_buffers` 字典 |

In [7]:
# ============================================================
# 示例：Parameter vs Buffer 训练中的不同行为
# ============================================================

class CompareParamBuffer(nn.Module):
    def __init__(self):
        super().__init__()
        # Parameter: 可训练，会被优化器更新
        self.weight = nn.Parameter(torch.randn(3, 3))
        # Buffer: 不可训练，但需要保存到 state_dict
        # 常用于 BatchNorm 的 running_mean 和 running_var
        self.register_buffer('running_mean', torch.zeros(3))
        # 普通 Tensor: 既不训练也不保存
        self.temp = torch.ones(3)
    def forward(self, x):
        # 前向传播中使用 Parameter 和 Buffer
        return x @ self.weight + self.running_mean

model = CompareParamBuffer()
optimizer = optim.SGD(model.parameters(), lr=0.01)

print("=" * 60)
print("【Parameter vs Buffer 训练中的不同行为】")
print("=" * 60)

print(f"\n1. 存储位置:")
print(f"   weight 在 _parameters 中: {'weight' in model._parameters}")
print(f"   running_mean 在 _buffers 中: {'running_mean' in model._buffers}")
print(f"   temp 不在任何容器中: {'temp' not in model._parameters and 'temp' not in model._buffers}")

print(f"\n2. 优化器管理的参数数量: {len(list(model.parameters()))}")
print(f"   → 只有 weight 被优化器管理")

# 前向传播 + 反向传播
x = torch.randn(2, 3)
loss = model(x).sum()  # 对输出求和作为损失
loss.backward()        # 反向传播

print(f"\n3. 反向传播后:")
print(f"   weight.grad 存在: {model.weight.grad is not None}")
print(f"   weight.grad 范数: {model.weight.grad.norm().item():.6f}")
print(f"   running_mean.grad 存在: {model.running_mean.grad is not None}")
print(f"   → Buffer 不参与梯度计算，因此 grad 为 None")

# 优化器 step
weight_before = model.weight.clone()
mean_before = model.running_mean.clone()
optimizer.step()  # 只更新 Parameter，不更新 Buffer

print(f"\n4. 优化器 step 后:")
print(f"   weight 变化: {(weight_before - model.weight).abs().sum().item() > 0}")
print(f"   weight 更新量: {(weight_before - model.weight).abs().sum().item():.6f}")
print(f"   running_mean 变化: {(mean_before - model.running_mean).abs().sum().item() == 0}")

print(f"\n5. state_dict 内容:")
print(f"   state_dict 键: {list(model.state_dict().keys())}")
print(f"   → 同时包含 Parameter 和 Buffer")
print(f"   → 普通 Tensor temp 不在 state_dict 中")

【Parameter vs Buffer 训练中的不同行为】

1. 存储位置:
   weight 在 _parameters 中: True
   running_mean 在 _buffers 中: True
   temp 不在任何容器中: True

2. 优化器管理的参数数量: 1
   → 只有 weight 被优化器管理

3. 反向传播后:
   weight.grad 存在: True
   weight.grad 范数: 5.111614
   running_mean.grad 存在: False
   → Buffer 不参与梯度计算，因此 grad 为 None

4. 优化器 step 后:
   weight 变化: True
   weight 更新量: 0.144408
   running_mean 变化: True

5. state_dict 内容:
   state_dict 键: ['weight', 'running_mean']
   → 同时包含 Parameter 和 Buffer
   → 普通 Tensor temp 不在 state_dict 中


## 7. `nn.Module` 共有方法

理解了状态存储之后，我们再来看 `nn.Module` 暴露给用户的所有方法，就能更清楚地理解每个方法背后在操作什么。

### 分类一：数据访问与遍历

| 方法 | 返回值 | 说明 |
|------|--------|------|
| `parameters()` | 可训练参数 | 传给优化器 |
| `named_parameters()` | (名称, 参数) | 调试、按名操作 |
| `buffers()` | Buffer 张量 | 获取统计量 |
| `named_buffers()` | (名称, Buffer) | 查看统计量 |
| `children()` | 直接子模块 | 仅遍历一级 |
| `named_children()` | (名称, 子模块) | 查看直接子模块 |
| `modules()` | 所有模块 | 递归遍历 |
| `named_modules()` | (名称, 模块) | 递归遍历 |
| `get_parameter()` | 指定参数 | 按路径查找 |
| `get_submodule()` | 指定子模块 | 按路径查找 |

In [8]:
# ============================================================
# 示例：数据访问与遍历方法
# ============================================================

class DemoModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 20)
        self.fc2 = nn.Linear(20, 5)
        self.relu = nn.ReLU()
        self.register_buffer('running_mean', torch.zeros(20))

model = DemoModel()

print("=" * 60)
print("【分类一：数据访问与遍历】")
print("=" * 60)

print("\n1. named_parameters():")
# named_parameters() 返回所有可训练参数的名称和值
for name, p in model.named_parameters():
    print(f"   {name}: {p.shape}")

params_list = list(model.parameters())
print(f"\n2. parameters(): {len(params_list)} 个参数张量")
# parameters() 只返回参数值，不包含名称，常用于传给优化器
for i, p in enumerate(params_list):
    print(f"   params[{i}]: {p.shape}")

print("\n3. named_buffers():")
# named_buffers() 返回所有注册的缓存张量
for name, buf in model.named_buffers():
    print(f"   {name}: {buf.shape}")

print("\n4. named_children(): (仅一级子模块)")
# children() 只返回直接的子模块，不会递归
for name, child in model.named_children():
    param_count = sum(p.numel() for p in child.parameters())
    print(f"   {name}: {child.__class__.__name__} (参数: {param_count})")

print("\n5. named_modules(): (递归遍历所有模块)")
# modules() 递归遍历所有子模块，包含自身
for name, module in model.named_modules():
    display_name = name if name else "(自身)"
    param_count = sum(p.numel() for p in module.parameters())
    print(f"   {display_name}: {module.__class__.__name__} (参数: {param_count})")

print(f"\n6. get_parameter('fc1.weight'): {model.get_parameter('fc1.weight').shape}")
# get_parameter() 通过完整路径名获取指定参数

print(f"\n7. get_submodule('fc2'): {model.get_submodule('fc2').__class__.__name__}")
# get_submodule() 通过完整路径名获取指定子模块

print("\n8. children() vs modules() 区别:")
print(f"   children() 数量: {len(list(model.children()))}")
print(f"   modules() 数量: {len(list(model.modules()))}")
print(f"   → modules() 包含自身，递归遍历所有子模块")

【分类一：数据访问与遍历】

1. named_parameters():
   fc1.weight: torch.Size([20, 10])
   fc1.bias: torch.Size([20])
   fc2.weight: torch.Size([5, 20])
   fc2.bias: torch.Size([5])

2. parameters(): 4 个参数张量
   params[0]: torch.Size([20, 10])
   params[1]: torch.Size([20])
   params[2]: torch.Size([5, 20])
   params[3]: torch.Size([5])

3. named_buffers():
   running_mean: torch.Size([20])

4. named_children(): (仅一级子模块)
   fc1: Linear (参数: 220)
   fc2: Linear (参数: 105)
   relu: ReLU (参数: 0)

5. named_modules(): (递归遍历所有模块)
   (自身): DemoModel (参数: 325)
   fc1: Linear (参数: 220)
   fc2: Linear (参数: 105)
   relu: ReLU (参数: 0)

6. get_parameter('fc1.weight'): torch.Size([20, 10])

7. get_submodule('fc2'): Linear

8. children() vs modules() 区别:
   children() 数量: 3
   modules() 数量: 4
   → modules() 包含自身，递归遍历所有子模块


### 分类二：模块管理

| 方法 | 操作 | 说明 |
|------|------|------|
| `add_module(name, module)` | 添加子模块 | 存入 `_modules` |

In [9]:
# ============================================================
# 示例：模块管理
# ============================================================

print("=" * 60)
print("【分类二：模块管理】")
print("=" * 60)

model = DemoModel()
print(f"\n初始 _modules 键: {list(model._modules.keys())}")

# add_module 方式添加子模块
# 这是注册子模块的显式方法，效果与属性赋值相同
model.add_module("dropout", nn.Dropout(0.5))
print(f"add_module 后: {list(model._modules.keys())}")

# 验证参数是否被追踪
param_count_before = sum(p.numel() for p in model.parameters())
print(f"添加后总参数: {param_count_before}")

# delattr 删除子模块
# 删除后该模块不再被追踪，参数也不会出现在 state_dict 中
delattr(model, "dropout")
print(f"delattr 后: {list(model._modules.keys())}")

# 直接属性赋值方式添加（最常用的方式）
# 这是 __setattr__ 自动分流机制在起作用
model.activation = nn.Tanh()
print(f"属性赋值后: {list(model._modules.keys())}")

# 验证：两种方式效果相同，都会存入 _modules
print(f"\n验证: activation 在 _modules 中? {'activation' in model._modules}")

【分类二：模块管理】

初始 _modules 键: ['fc1', 'fc2', 'relu']
add_module 后: ['fc1', 'fc2', 'relu', 'dropout']
添加后总参数: 325
delattr 后: ['fc1', 'fc2', 'relu']
属性赋值后: ['fc1', 'fc2', 'relu', 'activation']

验证: activation 在 _modules 中? True


### 分类三：注册 API

| 方法 | 存入容器 | 用途 |
|------|----------|------|
| `register_parameter(name, param)` | `_parameters` | 手动注册 Parameter |
| `register_buffer(name, tensor, persistent=True)` | `_buffers` | 注册非训练缓存 |
| `register_module(name, module)` | `_modules` | 底层注册子模块 |

---

既然 `__setattr__` 能自动分流，为什么还需要这三种注册方法？

**一句话回答**：自动分流是给"常规情况"用的，注册 API 是给"特殊情况"用的——处理自动分流无法覆盖的场景。

**逐一拆解**：

**1. `register_buffer()` —— 最常用，自动分流做不到**

```python
# ❌ 自动分流不会把普通 Tensor 存入 _buffers
self.running_mean = torch.zeros(3)   # 存入 __dict__，不会被保存

# ✅ 必须用 register_buffer 显式注册
self.register_buffer('running_mean', torch.zeros(3))  # 存入 _buffers
```

为什么需要？因为 `__setattr__` 只识别 `nn.Parameter` 和 `nn.Module`，普通 Tensor 会被忽略。Buffer 恰好是"普通 Tensor 但需要保存"，所以必须单独提供注册方法。

**2. `register_parameter()` —— 动态场景下使用**

```python
# 常规情况：直接赋值就够了
self.weight = nn.Parameter(torch.randn(3, 3))

# 特殊情况：参数名是动态的，无法写死在代码里
def add_param(self, name, shape):
    self.register_parameter(name, nn.Parameter(torch.randn(shape)))
```

为什么需要？当参数名是变量时，无法用 `self.xxx = ...` 硬编码，需要用 `register_parameter(name, param)` 动态注册。

**3. `register_module()` —— 底层实现，普通用户不用**

```python
# 常规情况：直接赋值
self.fc = nn.Linear(10, 5)

# 底层实现：register_module 是 add_module 的底层
# 普通用户不需要直接调用
```

为什么需要？这是 `add_module()` 和 `__setattr__` 的底层实现，框架内部使用。普通用户用 `self.fc = ...` 就够了。

---

**总结对比**：

| 方法 | 触发条件 | 使用场景 |
|------|---------|---------|
| `self.xxx = nn.Parameter()` | `__setattr__` 自动分流 | **常规注册参数**（最常用） |
| `self.xxx = nn.Module子类` | `__setattr__` 自动分流 | **常规注册子模块**（最常用） |
| `register_buffer()` | 必须显式调用 | **注册需要保存但不训练的统计量** |
| `register_parameter()` | 必须显式调用 | **动态创建参数**（参数名是变量） |
| `register_module()` | 必须显式调用 | **底层实现**，普通用户不需要 |

> 💡 **一句话总结**：`register_buffer()` 是三种注册 API 中最常用的，因为 `__setattr__` 不会把普通 Tensor 存入 `_buffers`，而 BatchNorm 的均值和方差又必须被保存。



In [10]:
# ============================================================
# 示例：注册 API
# ============================================================

class RegisterDemo(nn.Module):
    def __init__(self):
        super().__init__()
        # 1. register_parameter 注册可训练参数
        # 这是比属性赋值更显式的注册方式，常用于动态创建参数
        self.register_parameter('weight', nn.Parameter(torch.randn(3, 3)))
        self.register_parameter('bias', nn.Parameter(torch.zeros(3)))
        # 2. register_buffer 注册缓存 (persistent=True 默认)
        # Buffer 会出现在 state_dict 中，但不参与梯度计算
        self.register_buffer('running_mean', torch.zeros(3))
        # 3. register_buffer 注册临时缓存 (persistent=False, 不保存)
        # persistent=False 的 Buffer 不会出现在 state_dict 中
        self.register_buffer('temp_cache', torch.zeros(3), persistent=False)
        # 4. register_module 注册子模块
        # 这是 add_module 的底层实现，通常使用 add_module 或属性赋值即可
        self.register_module('fc', nn.Linear(5, 3))

demo = RegisterDemo()

print("=" * 60)
print("【分类三：注册 API】")
print("=" * 60)

print("\n1. _parameters 内容:")
for name, param in demo._parameters.items():
    print(f"   {name}: {param.shape} (requires_grad={param.requires_grad})")

print("\n2. _buffers 内容:")
for name, buf in demo._buffers.items():
    print(f"   {name}: {buf.shape}")

print("\n3. _modules 内容:")
for name, module in demo._modules.items():
    print(f"   {name}: {module.__class__.__name__}")

print(f"\n4. state_dict 键: {list(demo.state_dict().keys())}")
print("   注意: temp_cache 不在 state_dict 中 (persistent=False)")
print("   注意: fc 子模块的参数会被递归收集为 'fc.weight', 'fc.bias'")

【分类三：注册 API】

1. _parameters 内容:
   weight: torch.Size([3, 3]) (requires_grad=True)
   bias: torch.Size([3]) (requires_grad=True)

2. _buffers 内容:
   running_mean: torch.Size([3])
   temp_cache: torch.Size([3])

3. _modules 内容:
   fc: Linear

4. state_dict 键: ['weight', 'bias', 'running_mean', 'fc.weight', 'fc.bias']
   注意: temp_cache 不在 state_dict 中 (persistent=False)
   注意: fc 子模块的参数会被递归收集为 'fc.weight', 'fc.bias'


### 分类四：模式控制

model.train()：模型进入"训练状态"，允许调整那些需要在训练中更新的状态值。

model.eval()：模型进入"推断状态"，锁定这些状态值，直接用锁定后的值进行推断。

| 方法 | 作用 |
|------|------|
| `train(mode=True)` | 切换为训练模式 |
| `eval()` | 切换为推理模式 |
| `requires_grad_(requires_grad=True)` | 批量设置参数梯度 |

In [11]:
# ============================================================
# 示例：模式控制
# ============================================================

print("=" * 60)
print("【分类四：模式控制】")
print("=" * 60)

model = DemoModel()

# 1. train/eval 切换
# training 标志会影响 Dropout、BatchNorm 等层的行为
print(f"\n1. 初始 training: {model.training}")

model.eval()  # 切换到推理模式
print(f"   eval() 后: {model.training}")
print(f"   子模块 training 也同步变化: fc1.training={model.fc1.training}")

model.train()  # 切换回训练模式
print(f"   train() 后: {model.training}")

# 2. requires_grad 批量设置
# 当需要冻结整个模型时，可以用 requires_grad_(False)
first_param = next(model.parameters())
print(f"\n2. 初始 requires_grad: {first_param.requires_grad}")

model.requires_grad_(False)  # 冻结所有参数
print(f"   requires_grad_(False) 后: {first_param.requires_grad}")
print(f"   fc2.weight.requires_grad: {model.fc2.weight.requires_grad}")

model.requires_grad_(True)   # 解冻所有参数
print(f"   requires_grad_(True) 后: {first_param.requires_grad}")

# 3. train(mode=False) 等价于 eval()
model.train(mode=False)
print(f"\n3. train(mode=False) 后 training: {model.training}")
model.train(mode=True)
print(f"   train(mode=True) 后 training: {model.training}")

【分类四：模式控制】

1. 初始 training: True
   eval() 后: False
   子模块 training 也同步变化: fc1.training=False
   train() 后: True

2. 初始 requires_grad: True
   requires_grad_(False) 后: False
   fc2.weight.requires_grad: False
   requires_grad_(True) 后: True

3. train(mode=False) 后 training: False
   train(mode=True) 后 training: True


### 分类五：设备与数据类型迁移

| 方法 | 作用 |
|------|------|
| `to(device, dtype)` | 迁移到指定设备/精度 |
| `cpu()` | 迁移到 CPU |
| `cuda(device)` | 迁移到 GPU |
| `half()` | 转 float16 |
| `float()` | 转 float32 |
| `double()` | 转 float64 |
| `bfloat16()` | 转 bfloat16 |

In [12]:
# ============================================================
# 示例：设备与数据类型迁移
# ============================================================

print("=" * 60)
print("【分类五：设备与数据类型迁移】")
print("=" * 60)

model = DemoModel()

# 1. 数据类型转换
# 这些方法会 in-place 地修改模型中所有张量的数据类型
print(f"\n1. 初始 dtype:")
print(f"   fc1.weight.dtype: {model.fc1.weight.dtype}")
print(f"   running_mean.dtype: {model.running_mean.dtype}")

model.half()  # 转为 float16
print(f"\n   half() 后:")
print(f"   fc1.weight.dtype: {model.fc1.weight.dtype}")

model.float()  # 转为 float32
print(f"\n   float() 后:")
print(f"   fc1.weight.dtype: {model.fc1.weight.dtype}")

model.double()  # 转为 float64
print(f"\n   double() 后:")
print(f"   fc1.weight.dtype: {model.fc1.weight.dtype}")

model.float()  # 恢复为 float32

# 2. 设备迁移
# 当使用 GPU 训练时，需要先将模型移到 GPU 上
print(f"\n2. 当前设备: {model.fc1.weight.device}")
if torch.cuda.is_available():
    model.cuda()  # 移到默认 GPU
    print(f"   cuda() 后: {model.fc1.weight.device}")
    model.cpu()   # 移回 CPU
    print(f"   cpu() 后: {model.fc1.weight.device}")
else:
    print("   CUDA 不可用，跳过 GPU 迁移测试")

# 3. to() 方法统一迁移
# to() 是最通用的方法，可以同时指定设备和数据类型
print(f"\n3. to() 方法:")
model.to(dtype=torch.float64)
print(f"   to(dtype=float64) 后 dtype: {model.fc1.weight.dtype}")
model.float()  # 恢复

【分类五：设备与数据类型迁移】

1. 初始 dtype:
   fc1.weight.dtype: torch.float32
   running_mean.dtype: torch.float32

   half() 后:
   fc1.weight.dtype: torch.float16

   float() 后:
   fc1.weight.dtype: torch.float32

   double() 后:
   fc1.weight.dtype: torch.float64

2. 当前设备: cpu
   cuda() 后: cuda:0
   cpu() 后: cpu

3. to() 方法:
   to(dtype=float64) 后 dtype: torch.float64


DemoModel(
  (fc1): Linear(in_features=10, out_features=20, bias=True)
  (fc2): Linear(in_features=20, out_features=5, bias=True)
  (relu): ReLU()
)

### 分类六：梯度操作

| 方法 | 作用 |
|------|------|
| `zero_grad(set_to_none=False)` | 清空所有参数的梯度 |

In [13]:
# ============================================================
# 示例：梯度操作
# ============================================================

print("=" * 60)
print("【分类六：梯度操作】")
print("=" * 60)

# 修正：DemoModel 必须包含 forward() 方法
class DemoModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 20)
        self.fc2 = nn.Linear(20, 5)
        self.relu = nn.ReLU()
        self.register_buffer('running_mean', torch.zeros(20))
    
    # 必须实现 forward()，定义数据如何流过各层
    def forward(self, x):
        x = self.relu(self.fc1(x))
        return self.fc2(x)

model = DemoModel()
x = torch.randn(2, 10)
out = model(x)
out.sum().backward()  # 反向传播计算梯度

print(f"\n1. 反向传播后:")
print(f"   fc1.weight.grad 存在: {model.fc1.weight.grad is not None}")
print(f"   fc1.weight.grad 形状: {model.fc1.weight.grad.shape}")
print(f"   fc1.weight.grad 范数: {model.fc1.weight.grad.norm().item():.6f}")

# zero_grad() 默认行为
# PyTorch 2.0+ 默认将梯度置为 None 以节省内存
model.zero_grad()
print(f"\n2. zero_grad() 后:")
print(f"   fc1.weight.grad 是否为 None: {model.fc1.weight.grad is None}")
print(f"   → 默认将梯度置为 None (PyTorch 2.0+)")

# 再次反向传播测试
out = model(x)
out.sum().backward()

# set_to_none=True 显式置为 None
# 这是推荐的做法，可以节省内存
model.zero_grad(set_to_none=True)
print(f"\n3. zero_grad(set_to_none=True) 后:")
print(f"   fc1.weight.grad: {model.fc1.weight.grad}")
print(f"   → 显式将梯度置为 None")

# set_to_none=False 置为 0 张量
# 这种方式的优点是梯度张量可以复用，减少内存分配开销
out = model(x)
out.sum().backward()
model.zero_grad(set_to_none=False)
print(f"\n4. zero_grad(set_to_none=False) 后:")
print(f"   fc1.weight.grad 是否为 None: {model.fc1.weight.grad is None}")
print(f"   → 梯度被设置为 0 张量")

【分类六：梯度操作】

1. 反向传播后:
   fc1.weight.grad 存在: True
   fc1.weight.grad 形状: torch.Size([20, 10])
   fc1.weight.grad 范数: 5.353317

2. zero_grad() 后:
   fc1.weight.grad 是否为 None: True
   → 默认将梯度置为 None (PyTorch 2.0+)

3. zero_grad(set_to_none=True) 后:
   fc1.weight.grad: None
   → 显式将梯度置为 None

4. zero_grad(set_to_none=False) 后:
   fc1.weight.grad 是否为 None: False
   → 梯度被设置为 0 张量


### 分类七：序列化
序列化就是把内存中分散的模型数据（权重、偏置、Buffer 等），按顺序排列成连续的字节流保存到硬盘；反序列化就是把硬盘上的字节流读回内存，恢复成可用的模型对象。

| 方法 | 作用 |
|------|------|
| `state_dict()` | 导出所有参数 + Buffer |
| `load_state_dict(state_dict, strict=True)` | 加载权重字典 |

In [14]:
# ============================================================
# 示例：序列化
# ============================================================

print("=" * 60)
print("【分类七：序列化】")
print("=" * 60)

model = DemoModel()
sd = model.state_dict()

print(f"\n1. state_dict 键: {list(sd.keys())}")
print(f"   state_dict 包含参数和 buffer")

# 2. 完整加载
# 创建新模型并加载完整的 state_dict
model_copy = DemoModel()
load_res = model_copy.load_state_dict(sd)
print(f"\n2. load_state_dict 结果: {load_res}")
print(f"   → 完整加载成功，所有键匹配")

# 3. strict=False 部分加载
# 当模型结构不完全匹配时，可以使用 strict=False 加载可用的权重
class SubModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 20)
        # 注意：没有 fc2、relu、running_mean

partial_model = SubModel()
load_res = partial_model.load_state_dict(sd, strict=False)
print(f"\n3. strict=False 加载 (只加载 fc1):")
print(f"   missing_keys: {load_res.missing_keys}")  # 目标模型有但权重中没有的键
print(f"   unexpected_keys: {load_res.unexpected_keys}")  # 权重中有但目标模型没有的键

# 4. 验证加载后的权重是否一致
print(f"\n4. 验证加载正确性:")
print(f"   原始 fc1.weight 均值: {model.fc1.weight.mean().item():.6f}")
print(f"   加载后 fc1.weight 均值: {partial_model.fc1.weight.mean().item():.6f}")
print(f"   是否一致: {torch.allclose(model.fc1.weight, partial_model.fc1.weight)}")

【分类七：序列化】

1. state_dict 键: ['running_mean', 'fc1.weight', 'fc1.bias', 'fc2.weight', 'fc2.bias']
   state_dict 包含参数和 buffer

2. load_state_dict 结果: <All keys matched successfully>
   → 完整加载成功，所有键匹配

3. strict=False 加载 (只加载 fc1):
   missing_keys: []
   unexpected_keys: ['running_mean', 'fc2.weight', 'fc2.bias']

4. 验证加载正确性:
   原始 fc1.weight 均值: -0.005594
   加载后 fc1.weight 均值: -0.005594
   是否一致: True


### 分类八：工具方法

本身不参与模型的前向传播或训练，但在模型初始化和模型调试时非常实用。


 `apply(fn)` ：递归对所有子模块执行函数

 `__repr__()` ：返回模型的层次结构和参数配置的可读字符串表示（从对象的内部属性中提取信息，按固定格式拼接而成），在 print() 或交互式环境中自动调用，用于展示模型结构和参数信息。

In [15]:
# ============================================================
# 示例：工具方法
# ============================================================

print("=" * 60)
print("【分类八：工具方法】")
print("=" * 60)

model = DemoModel()

# 1. apply() 统一初始化
# apply 会递归遍历所有子模块，对每个模块执行指定的函数
# 这是进行权重统一初始化的标准方式
def init_weights(m):
    if isinstance(m, nn.Linear):
        # 对 Linear 层使用 Kaiming 初始化
        nn.init.kaiming_normal_(m.weight)
        if m.bias is not None:
            # 偏置初始化为 0
            nn.init.zeros_(m.bias)
            print(f"   → 初始化 {m.__class__.__name__}: weight 用 kaiming_normal, bias 置零")

print(f"\n1. 初始化前 fc1.weight 范数: {model.fc1.weight.norm().item():.4f}")
print("   执行 model.apply(init_weights):")
model.apply(init_weights)   #apply就是遍历所有子模块的，记住这个约定
print(f"   初始化后 fc1.weight 范数: {model.fc1.weight.norm().item():.4f}")
print(f"   fc1.bias 是否全零: {torch.all(model.fc1.bias == 0)}")


【分类八：工具方法】

1. 初始化前 fc1.weight 范数: 2.4966
   执行 model.apply(init_weights):
   → 初始化 Linear: weight 用 kaiming_normal, bias 置零
   → 初始化 Linear: weight 用 kaiming_normal, bias 置零
   初始化后 fc1.weight 范数: 6.4913
   fc1.bias 是否全零: True


In [16]:
model = DemoModel()

# 3. __repr__
# 打印模型的字符串表示，显示结构层次
print(f"\n3. __repr__ 输出:\n{model}")


3. __repr__ 输出:
DemoModel(
  (fc1): Linear(in_features=10, out_features=20, bias=True)
  (fc2): Linear(in_features=20, out_features=5, bias=True)
  (relu): ReLU()
)


### 分类九：钩子函数

| 方法 | 触发时机 | 用途 |
|------|----------|------|
| `register_forward_pre_hook(hook)` | `forward()` 执行之前 | 修改输入 |
| `register_forward_hook(hook)` | `forward()` 执行之后 | 获取中间层输出 |
| `register_full_backward_hook(hook)` | 反向传播时 | 修改梯度 |

In [17]:
# ============================================================
# 示例：钩子函数
# ============================================================

print("=" * 60)
print("【分类九：钩子函数】")
print("=" * 60)

# 用于存储钩子捕获的数据
activation = {'input': None, 'output': None}

# 前向预钩子：在 forward 执行前触发，可用于修改输入
def forward_pre_hook(module, input):
    activation['input'] = input[0].detach()
    print(f"   Forward Pre-Hook: {module.__class__.__name__}")
    print(f"      输入形状: {input[0].shape}")
    print(f"      输入均值: {input[0].mean().item():.4f}")

# 前向钩子：在 forward 执行后触发，可用于获取中间层输出
def forward_hook(module, input, output):
    activation['output'] = output.detach()
    print(f"   Forward Hook: {module.__class__.__name__}")
    print(f"      输出形状: {output.shape}")
    print(f"      输出均值: {output.mean().item():.4f}")

model = DemoModel()

# 注册钩子到 fc1 层
# register_forward_pre_hook 和 register_forward_hook 返回句柄
handle_pre = model.fc1.register_forward_pre_hook(forward_pre_hook)
handle_post = model.fc1.register_forward_hook(forward_hook)

print("\n执行前向传播 (x 经过 fc1 时触发钩子):")
x = torch.randn(2, 10)
out = model(x)

print(f"\n钩子捕获的数据:")
print(f"   fc1 输入形状: {activation['input'].shape}")
print(f"   fc1 输出形状: {activation['output'].shape}")

# 清理钩子
# 使用完毕需要移除钩子，避免内存泄漏
handle_pre.remove()
handle_post.remove()
print(f"\n钩子已移除")

【分类九：钩子函数】

执行前向传播 (x 经过 fc1 时触发钩子):
   Forward Pre-Hook: Linear
      输入形状: torch.Size([2, 10])
      输入均值: -0.5426
   Forward Hook: Linear
      输出形状: torch.Size([2, 20])
      输出均值: -0.0669

钩子捕获的数据:
   fc1 输入形状: torch.Size([2, 10])
   fc1 输出形状: torch.Size([2, 20])

钩子已移除


### 分类十：魔法方法

这些方法你不需要显式调用它们，它们是在特定操作发生时自动触发的，就像被施了魔法一样。

```python
普通方法：你主动调用
model.forward(x)     # ← 你主动敲代码调用
model.zero_grad()    # ← 你主动敲代码调用

魔法方法：你写一个操作，它自动触发
model(x)             # ← 你只是调用模型，__call__ 和 forward 自动执行
print(model)         # ← 你只是打印，__repr__ 自动执行
self.fc = Linear()   # ← 你只是赋值，__setattr__ 自动执行
```

| 方法 | 作用 | 何时调用 |
|------|------|----------|
| `forward(x)` | 前向传播逻辑 | `model(x)` 内部调用 |
| `__call__(x)` | 封装 forward + 钩子 | `model(x)` 时自动触发 |
| `__setattr__` | 自动分流 | `self.xxx = ...` 时触发 |
| `__repr__` | 格式化输出 | `print(model)` 时触发 |

In [18]:
# ============================================================
# 示例：魔法方法
# ============================================================

class MagicDemo(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 5)
        self._call_count = 0
    
    # forward 定义了模块的核心计算逻辑
    # 当调用 model(x) 时，__call__ 会调用 forward
    def forward(self, x):
        print("   → forward() 被调用")
        self._call_count += 1
        return self.fc(x)

model = MagicDemo()
x = torch.randn(3, 10)

print("=" * 60)
print("【分类十：魔法方法】")
print("=" * 60)

print("\n1. 调用 model(x):")
# __call__ 会先执行钩子逻辑，再调用 forward
out = model(x)
print(f"   _call_count: {model._call_count}")


【分类十：魔法方法】

1. 调用 model(x):
   → forward() 被调用
   _call_count: 1


In [19]:
print("\n2. 直接调用 forward(x):")
# 直接调用 forward 会跳过 __call__ 中的钩子逻辑
out = model.forward(x)
print(f"   _call_count: {model._call_count}")
print("   → 直接调用 forward 不会触发 __call__ 中的钩子逻辑")


2. 直接调用 forward(x):
   → forward() 被调用
   _call_count: 2
   → 直接调用 forward 不会触发 __call__ 中的钩子逻辑


In [20]:
print("\n3. __setattr__ 自动分流:")
# 赋值 nn.Parameter 时自动存入 _parameters
model.new_param = nn.Parameter(torch.ones(3))
print(f"   new_param 在 _parameters 中: {'new_param' in model._parameters}")


3. __setattr__ 自动分流:
   new_param 在 _parameters 中: True


In [21]:
# 赋值 nn.Module 时自动存入 _modules
model.new_module = nn.ReLU()
print(f"   new_module 在 _modules 中: {'new_module' in model._modules}")

   new_module 在 _modules 中: True


In [22]:
# 赋值普通 Python 值存入 __dict__
model.new_value = 42
print(f"   new_value 在 __dict__ 中: {'new_value' in model.__dict__}")

   new_value 在 __dict__ 中: True


In [23]:
print("\n4. __repr__ 打印:")
# __repr__ 提供了友好的模型结构展示
print(repr(model))


4. __repr__ 打印:
MagicDemo(
  (fc): Linear(in_features=10, out_features=5, bias=True)
  (new_module): ReLU()
)


## 8. 总论总结

### 核心概念速查表

| 概念 | 定义 | 典型代表 |
|------|------|----------|
| **`nn.Module`** | 所有神经网络组件的基类 | 所有层、模型、容器的父类 |
| **网络层** | 执行数据变换的计算单元 | Linear, Conv2d, ReLU |
| **`nn.functional`** | 无状态的纯函数 | F.relu, F.conv2d |
| **单一层** | 原子操作，不可再分 | Conv2d, Linear |
| **复合层** | 多个子层组合 | LSTM, TransformerEncoder |
| **有状态层** | 有可训练参数 | Linear, Conv2d |
| **无参数层** | 无参数但有状态 | ReLU, Dropout |
| **无状态函数** | 纯函数 | F.relu, F.dropout |
| **Parameter** | 可训练，优化器更新 | weight, bias |
| **Buffer** | 不可训练，但需保存 | running_mean, running_var |

### 选择指南

| 场景 | 推荐 |
|------|------|
| 需要参数训练 | 使用 `nn.Module` 子类（有状态层） |
| 只需计算，无参数 | 用 `F` 函数（轻量）或 `nn.Module` 子类（放容器） |
| 放入 `nn.Sequential` | 必须使用 `nn.Module` 子类 |
| 保存和加载模型 | 使用 `state_dict()` / `load_state_dict()` |
| GPU 训练 | 使用 `.to(device)` / `.cuda()` |
| 权重统一初始化 | 使用 `.apply(fn)` |
| 特征可视化 | 使用 `register_forward_hook()` |
| 模型深度拷贝 | 使用 `copy.deepcopy(model)` |

**下一部分预告**：第2部分将详细介绍所有**有参数层**（卷积、线性、归一化、嵌入、序列层、Transformer 核心组件）。